# Product Category Classifier

**Autor:** Nikola Kostić

## Cilj projekta

Cilj je automatska klasifikacija e-commerce proizvoda na osnovu `Product Title` vrednosti. Projekat prati kompletan ML workflow: analiza podataka, čišćenje, feature engineering, poređenje modela, evaluacija i izbor finalnog modela.

In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


## 1. Učitavanje i pregled dataseta

Prvo proveravamo strukturu, nedostajuće vrednosti i distribuciju ciljnih kategorija.

In [ ]:
df = pd.read_csv("products.csv")

# U izvornom fajlu neke kolone imaju skrivene razmake.
df.columns = [column.strip() for column in df.columns]

print("Shape:", df.shape)
display(df.head())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))

print("\nOriginal category counts:")
display(df["Category Label"].value_counts(dropna=False).to_frame("count"))


### Zapažanje

Dataset sadrži praktične probleme kakvi su česti u realnim projektima:

- nedostajuće naslove/kategorije;
- vodeće i završne razmake u nazivima kolona;
- različita imena za istu kategoriju (`CPU` / `CPUs`, `Mobile Phone` / `Mobile Phones`, `fridge` / `Fridges`).

Pre modeliranja ih standardizujemo.

In [ ]:
data = df.dropna(
    subset=["Product Title", "Category Label"]
).copy()

data["Product Title"] = (
    data["Product Title"]
    .astype(str)
    .str.strip()
    .str.lower()
)

data["Category Label"] = (
    data["Category Label"]
    .astype(str)
    .str.strip()
)

data["Category Label"] = data["Category Label"].replace({
    "fridge": "Fridges",
    "CPU": "CPUs",
    "Mobile Phone": "Mobile Phones",
})

data = data[data["Product Title"] != ""]

print("Rows after cleaning:", len(data))
print("Categories:", data["Category Label"].nunique())
display(data["Category Label"].value_counts().to_frame("count"))


## 2. Jednostavan feature engineering

Pored samog teksta, istražujemo nekoliko osobina naslova: broj karaktera, broj reči, broj cifara i broj specijalnih karaktera. One mogu biti korisne jer tehnički proizvodi često sadrže model brojeve i kratke kodove.

Finalni model ipak koristi word + character TF-IDF, jer je taj pristup dao najbolji rezultat i direktno hvata ovakve obrasce.

In [ ]:
data["title_char_count"] = data["Product Title"].str.len()
data["title_word_count"] = data["Product Title"].str.split().str.len()
data["digit_count"] = data["Product Title"].str.count(r"\d")
data["special_char_count"] = data["Product Title"].str.count(r"[^a-z0-9\s]")

display(
    data[
        [
            "Product Title",
            "title_char_count",
            "title_word_count",
            "digit_count",
            "special_char_count",
            "Category Label",
        ]
    ].head()
)

display(
    data.groupby("Category Label")[
        ["title_char_count", "title_word_count", "digit_count"]
    ].mean().round(2)
)


## 3. Distribucija kategorija

In [ ]:
category_counts = data["Category Label"].value_counts()

plt.figure(figsize=(10, 6))
plt.barh(category_counts.index, category_counts.values)
plt.title("Broj proizvoda po kategoriji")
plt.xlabel("Broj proizvoda")
plt.ylabel("Kategorija")
plt.tight_layout()
plt.show()


## 4. Train/test split

Koristimo stratifikovan 80/20 split sa `random_state=42`, kako bi sve kategorije imale sličnu zastupljenost u trening i test skupu.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["Product Title"],
    data["Category Label"],
    test_size=0.20,
    random_state=42,
    stratify=data["Category Label"],
)

print("Train:", len(X_train))
print("Test:", len(X_test))


## 5. Logistic Regression i Multinomial Naive Bayes

Prva dva modela koriste word TF-IDF sa unigramima i bigramima.

In [ ]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_test_word = word_vectorizer.transform(X_test)

logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42,
)

nb_model = MultinomialNB()

logistic_model.fit(X_train_word, y_train)
nb_model.fit(X_train_word, y_train)

pred_logistic = logistic_model.predict(X_test_word)
pred_nb = nb_model.predict(X_test_word)


## 6. Linear SVC sa word + character TF-IDF

Kod proizvoda model brojevi i kratki kodovi nose dosta informacije. Zato uz reči dodajemo i character n-grame. Ovo je posebno korisno za nazive kao `KGV39VL31G` ili `SBS8004PO`.

In [ ]:
combined_features = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
        ),
    ),
    (
        "char",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True,
            max_features=80000,
        ),
    ),
])

X_train_combined = combined_features.fit_transform(X_train)
X_test_combined = combined_features.transform(X_test)

svc_model = LinearSVC(
    class_weight="balanced",
    random_state=42,
)

svc_model.fit(X_train_combined, y_train)
pred_svc = svc_model.predict(X_test_combined)


## 7. Poređenje modela

In [ ]:
predictions = {
    "Logistic Regression": pred_logistic,
    "Multinomial Naive Bayes": pred_nb,
    "Linear SVC (word + char TF-IDF)": pred_svc,
}

results = []

for model_name, predictions_for_model in predictions.items():
    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions_for_model),
        "macro_f1": f1_score(
            y_test,
            predictions_for_model,
            average="macro",
        ),
        "weighted_f1": f1_score(
            y_test,
            predictions_for_model,
            average="weighted",
        ),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

display(
    results_df.style.format({
        "accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "weighted_f1": "{:.4f}",
    })
)


## 8. Classification report za najbolji model

In [ ]:
print(
    classification_report(
        y_test,
        pred_svc,
        digits=4,
        zero_division=0,
    )
)


## 9. Confusion matrices

Matrice zabune pomažu da vidimo između kojih kategorija dolazi do najvećeg broja grešaka.

### Logistic Regression

In [ ]:
labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    pred_logistic,
    labels=labels,
)

fig, ax = plt.subplots(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels,
)

disp.plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False,
)

ax.set_title("Confusion Matrix - Logistic Regression")
plt.tight_layout()
plt.show()


### Multinomial Naive Bayes

In [ ]:
labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    pred_nb,
    labels=labels,
)

fig, ax = plt.subplots(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels,
)

disp.plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False,
)

ax.set_title("Confusion Matrix - Multinomial Naive Bayes")
plt.tight_layout()
plt.show()


### Linear SVC

In [ ]:
labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    pred_svc,
    labels=labels,
)

fig, ax = plt.subplots(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels,
)

disp.plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False,
)

ax.set_title("Confusion Matrix - Linear SVC")
plt.tight_layout()
plt.show()


## 10. Test proizvodi iz zadatka

Finalni model se trenira na svim očišćenim podacima, jer je evaluacija modela već završena na holdout test skupu.

In [ ]:
final_model = joblib.load("product_category_model.pkl")

test_products = [
    "iphone 7 32gb gold,4,3,Apple iPhone 7 32GB",
    "olympus e m10 mark iii geh use silber",
    "kenwood k20mss15 solo",
    "bosch wap28390gb 8kg 1400 spin",
    "bosch serie 4 kgv39vl31g",
    "smeg sbs8004po",
]
expected = [
    "Mobile Phones",
    "Digital Cameras",
    "Microwaves",
    "Washing Machines",
    "Fridge Freezers",
    "Fridge Freezers",
]

pred = final_model.predict([x.lower() for x in test_products])
manual = pd.DataFrame({
    "Product Title": test_products,
    "Prediction": pred,
    "Expected": expected,
})
manual["Correct"] = manual["Prediction"] == manual["Expected"]
display(manual)

## 11. Zaključak

Najbolji model je **Linear SVC sa kombinovanim word + character TF-IDF karakteristikama**.

Na 80/20 stratifikovanom test skupu postignuto je približno:

- **Accuracy: 0.9909**
- **Macro F1: 0.9905**
- **Weighted F1: 0.9909**

Linear SVC je značajno nadmašio Logistic Regression i Multinomial Naive Bayes. Najveći dobitak je došao iz character n-grama, jer produktni naslovi često sadrže kratke model kodove koji su veoma informativni.

### Šta nije bilo idealno iz prve?

Kod inicijalnog holdout testa, vrlo kratak naslov `smeg sbs8004po` mogao je biti teži za klasifikaciju kada konkretan model broj nije bio prisutan u trening delu splita. Nakon izbora modela, finalni model je po standardnoj praksi treniran nad **svim očišćenim podacima**, pa pravilno koristi postojeće informacije o tom model kodu.

### Dalje unapređenje

Sledeći koraci bi bili:

- cross-validation i hyperparameter tuning;
- analiza najčešćih confusion parova;
- praćenje novih kategorija i vocabulary drift-a;
- eventualno uključivanje drugih pouzdanih atributa proizvoda, ako su dostupni pri predikciji.
